# LSTM-GAT Trajectory End-to-End

Trajectory-based LSTM-GAT: concatenate all LSTM hidden states per stock,
compute attention once on the full trajectory vectors, predict one position per stock.

**Key difference from per-timestep GAT:** Loss is computed only on day 20's prediction
(the fully-informed one), not all 20 timesteps.

**Architecture**: Shared LSTM → Flatten 20×10=200 dims per stock → Full GAT → Position

## 1. Setup

In [ ]:
!pip install -q tensorflow>=2.16.0 keras-tuner empyrical-reloaded spektral

In [ ]:
import os
import sys

if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/repo'):
        !git clone https://github.com/adam-909/4yp.git /content/repo
    else:
        !cd /content/repo && git pull
    os.chdir('/content/repo/4YP-main')
else:
    os.chdir('/home/adam/new4YP/4YP-main')

sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from empyrical import (
    sharpe_ratio, sortino_ratio, max_drawdown,
    annual_return, annual_volatility, calmar_ratio,
)

import random
random.seed(40)
np.random.seed(40)

import tensorflow as tf
tf.random.set_seed(40)

print(f"TensorFlow version: {tf.__version__}")

## 2. Configuration

In [ ]:
# Training/Test Configuration
TRAIN_START = 2011
TEST_START = 2017
TEST_END = 2023

VOL_TARGET = 0.15

# Stride Configuration
# Controls overlap between training windows:
#   TRAIN_STRIDE = 20 → non-overlapping (57 samples, matches paper)
#   TRAIN_STRIDE = 10 → 50% overlap (~140 samples)
#   TRAIN_STRIDE = 5  → 75% overlap (~280 samples)
#   TRAIN_STRIDE = 1  → full sliding (~1400 samples)
TRAIN_STRIDE = 20

# Model Configuration
TOTAL_TIME_STEPS = 20
TRAIN_VALID_RATIO = 0.8
NUM_EPOCHS = 300
EARLY_STOPPING_PATIENCE = 25

# GAT Hyperparameters
HIDDEN_LAYER_SIZE = 10
GAT_UNITS = 8
ATTN_HEADS = 2
DROPOUT_RATE = 0.5
LEARNING_RATE = 0.0005
MAX_GRADIENT_NORM = 0.01
NUM_GAT_LAYERS = 1
BATCH_SIZE = 32

print(f"Train: {TRAIN_START}-{TEST_START}")
print(f"Test:  {TEST_START}-{TEST_END}")
print(f"\nModel: LSTM-GAT Trajectory End-to-End")
print(f"  Stride: {TRAIN_STRIDE}")
print(f"  GAT units: {GAT_UNITS}, heads: {ATTN_HEADS}, layers: {NUM_GAT_LAYERS}")
print(f"  Dropout: {DROPOUT_RATE}, LR: {LEARNING_RATE}")

## 3. Helper Functions

In [ ]:
# Training/Test Configuration
TRAIN_START = 2011
TEST_START = 2017
TEST_END = 2023

VOL_TARGET = 0.15  # 15% volatility target for normalization

# Model Configuration
TOTAL_TIME_STEPS = 20
TRAIN_VALID_RATIO = 0.8
NUM_EPOCHS = 300
EARLY_STOPPING_PATIENCE = 25

# GAT Hyperparameters
HIDDEN_LAYER_SIZE = 10
GAT_UNITS = 8        # Output dimension per attention head
ATTN_HEADS = 2       # Number of attention heads
DROPOUT_RATE = 0.5
LEARNING_RATE = 0.0005
MAX_GRADIENT_NORM = 0.01
NUM_GAT_LAYERS = 2
BATCH_SIZE = 32

print(f"Train: {TRAIN_START}-{TEST_START}")
print(f"Test:  {TEST_START}-{TEST_END}")
print(f"\nModel: LSTM-GAT End-to-End (Full Attention)")
print(f"  LSTM hidden: {HIDDEN_LAYER_SIZE}")
print(f"  GAT units: {GAT_UNITS}, heads: {ATTN_HEADS}")
print(f"  GAT layers: {NUM_GAT_LAYERS}")
print(f"  Dropout: {DROPOUT_RATE}")
print(f"  Learning rate: {LEARNING_RATE}")

## 4. Data Loading

In [ ]:
features_path = "data/straddle_features/features.csv"
df = pd.read_csv(features_path)
df['date'] = pd.to_datetime(df['date'])

print(f"Loaded {len(df)} rows")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Tickers: {df['ticker'].nunique()}")

In [ ]:
from gml.graph_model_inputs import GraphModelFeatures

features = GraphModelFeatures(
    df=df,
    total_time_steps=TOTAL_TIME_STEPS,
    start_boundary=TRAIN_START,
    test_boundary=TEST_START,
    test_end=TEST_END,
    train_valid_ratio=TRAIN_VALID_RATIO,
    split_tickers_individually=True,
    time_features=False,
    train_valid_sliding=True,  # Enable sliding windows for stride control
)

print("Feature generator created (sliding windows enabled).")

In [ ]:
# Apply stride to training/validation data
# TRAIN_STRIDE=20 → non-overlapping (matches paper)
# TRAIN_STRIDE=1  → full sliding (maximum data)
train_data = {k: v[::TRAIN_STRIDE] for k, v in features.train.items()}
valid_data = {k: v[::TRAIN_STRIDE] for k, v in features.valid.items()}
test_data = features.test_sliding  # Test always uses stride=1

print(f"Stride: {TRAIN_STRIDE}")
print(f"\nTraining data:")
print(f"  inputs: {train_data['inputs'].shape}")
print(f"  outputs: {train_data['outputs'].shape}")
print(f"\nValidation data:")
print(f"  inputs: {valid_data['inputs'].shape}")
print(f"\nTest data:")
print(f"  inputs: {test_data['inputs'].shape}")

## 5. Model Definition

In [ ]:
from gml.graph_model_gat_v2 import build_lstm_gat_trajectory_model

num_tickers = train_data['inputs'].shape[1]
time_steps = train_data['inputs'].shape[2]
input_size = train_data['inputs'].shape[3]

print(f"Building LSTM-GAT Trajectory E2E model:")
print(f"  num_tickers: {num_tickers}")
print(f"  time_steps: {time_steps} (concat to {time_steps * HIDDEN_LAYER_SIZE} dims)")
print(f"  input_size: {input_size}")

model = build_lstm_gat_trajectory_model(
    num_tickers=num_tickers, time_steps=time_steps, input_size=input_size,
    hidden_layer_size=HIDDEN_LAYER_SIZE, gat_units=GAT_UNITS, attn_heads=ATTN_HEADS,
    dropout_rate=DROPOUT_RATE, learning_rate=LEARNING_RATE,
    max_gradient_norm=MAX_GRADIENT_NORM, num_gat_layers=NUM_GAT_LAYERS,
)

# model.summary()

## 6. Training

In [ ]:
# Extract last timestep labels only (trajectory predicts day 20 only)
X_train = train_data['inputs']
y_train = train_data['outputs'][:, :, -1:, :]  # (num_windows, 88, 1, 1)
y_train = y_train.squeeze(-1)  # (num_windows, 88, 1)
w_train = train_data['active_entries'][:, :, -1:, :].squeeze(-1)

X_valid = valid_data['inputs']
y_valid = valid_data['outputs'][:, :, -1:, :].squeeze(-1)
w_valid = valid_data['active_entries'][:, :, -1:, :].squeeze(-1)

print(f"Training: X={X_train.shape}, y={y_train.shape}")
print(f"Validation: X={X_valid.shape}, y={y_valid.shape}")

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=EARLY_STOPPING_PATIENCE,
    restore_best_weights=True,
    verbose=1
)

print("="*60)
print("Training LSTM-GAT Trajectory E2E")
print(f"  Stride: {TRAIN_STRIDE} ({X_train.shape[0]} training samples)")
print(f"  GAT units: {GAT_UNITS}, heads: {ATTN_HEADS}, layers: {NUM_GAT_LAYERS}")
print("="*60)

history = model.fit(
    X_train, y_train,
    sample_weight=w_train,
    validation_data=(X_valid, y_valid, w_valid),
    epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=1,
)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (Negative Sharpe)')
plt.title('LSTM-GAT Trajectory E2E Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Epochs trained: {len(history.history['loss'])}")
print(f"Best val loss: {min(history.history['val_loss']):.4f}")

## 7. Evaluation

In [ ]:
X_test = test_data['inputs']
predictions = model.predict(X_test)

print(f"Predictions shape: {predictions.shape}")
print(f"Test outputs shape: {test_data['outputs'].shape}")

In [ ]:
# Trajectory model outputs (batch, 88, 1) — one prediction per stock per window
positions = predictions[:, :, 0].reshape(-1)
returns = test_data['outputs'][:, :, -1, 0].reshape(-1)
captured_returns = positions * returns

dates = test_data['date'][:, :, -1, 0].reshape(-1)
identifiers = test_data['identifier'][:, :, -1, 0].reshape(-1)

results_df = pd.DataFrame({
    'time': dates, 'identifier': identifiers,
    'position': positions, 'returns': returns,
    'captured_returns': captured_returns,
})

results_df['time'] = pd.to_datetime(results_df['time'])
results_df = results_df[results_df['identifier'] != '0']

print(f"Results: {len(results_df)} rows")
results_df.head()

In [ ]:
daily_returns = calc_daily_returns(results_df)

print("\n" + "="*60)
print("LSTM-GAT Traj E2E Results (Raw)")
print("="*60)

metrics_raw = calc_metrics(daily_returns, "LSTM-GAT Traj E2E")
display_metrics(metrics_raw)

print(f"\nVolatility-Normalized (Target: {VOL_TARGET:.0%})")
metrics_norm, scaled_returns = calc_metrics_vol_normalized(daily_returns, "LSTM-GAT Traj E2E", VOL_TARGET)
display_metrics(metrics_norm)

print("\nYearly Sharpe Ratios:")
yearly_sharpes = calc_yearly_sharpes(daily_returns)
for year, sharpe_val in yearly_sharpes.items():
    print(f"  {year}: {sharpe_val:.4f}")

In [ ]:
all_daily_returns = {'LSTM-GAT Traj E2E': daily_returns}
plot_results(all_daily_returns, "LSTM-GAT Traj E2E ({TEST_START}-{TEST_END})")

## 8. Position Analysis

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(results_df['position'], bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Position')
plt.ylabel('Frequency')
plt.title('Position Distribution')
plt.axvline(x=0, color='red', linestyle='--', linewidth=1)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Save Results

In [ ]:
results_dir = f"results/lstm_gat_trajectory_e2e_s{TRAIN_STRIDE}_u{GAT_UNITS}_h{ATTN_HEADS}/{TEST_START}-{TEST_END}"
os.makedirs(results_dir, exist_ok=True)

results_df.to_csv(os.path.join(results_dir, "captured_returns_sw.csv"), index=False)
pd.DataFrame([metrics_raw]).to_csv(os.path.join(results_dir, "metrics_raw.csv"), index=False)
pd.DataFrame([metrics_norm]).to_csv(os.path.join(results_dir, "metrics_vol_normalized.csv"), index=False)
pd.DataFrame(yearly_sharpes.items(), columns=['Year', 'Sharpe']).to_csv(os.path.join(results_dir, "yearly_sharpes.csv"), index=False)

print(f"Results saved to: {results_dir}")